
# LV 几何 AHA 17 分区可视化（PyVista 学术出图版）

这个 notebook 适合直接在 `.ipynb` 中运行，目标是：

1. 读取 Abaqus 六面体网格 `.inp`
2. 读取 MATLAB 分区标签 `.mat`
3. 构造 `PyVista UnstructuredGrid`
4. 输出更适合论文或报告的 **双视图静态图**：
   - 左：外表面 17 分区
   - 右：纵向裁切后的内部视图

> 说明  
> - 该实现是 **纯 Python** 工作流，不依赖 ParaView GUI。  
> - 更适合后续和 `numpy / scipy / matplotlib` 的研究代码整合。  
> - 默认使用 **off-screen rendering**，避免 notebook 里的交互后端问题。



## 运行前准备

建议在当前 notebook 环境中安装：

```bash
pip install pyvista vtk scipy matplotlib h5py
```

如果你已经在包含 `pyvista` 和 `vtk` 的环境中运行，可以跳过。


In [ ]:

import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.io import loadmat

try:
    import h5py   # 仅用于 MATLAB v7.3 文件回退读取
except Exception:
    h5py = None

import pyvista as pv


In [ ]:

# =========================
# 1) 用户配置：只需修改这里
# =========================
inp_path = "hexheartLVmeshF60S45.inp"
mat_path = "AHALVMeshDivisionPCAReconstructed.mat"

# 输出路径
output_dir = "output_dir"
figure_name = "lv_aha17_pyvista_academic.png"
vtu_name = "lv_aha17_mesh.vtu"   # 可选：同时导出 VTU，便于后续复用
save_vtu = True

# 图像参数
window_size = (2400, 1100)   # 渲染窗口大小
image_scale = 2              # 截图放大倍数；最终分辨率 = window_size * image_scale
show_edges_left = False      # 外表面一般不显示边线，更简洁
show_edges_right = True      # 裁切图显示边线，有助于看清分区边界
parallel_projection = True   # 学术静态图通常更适合平行投影


In [ ]:

# =========================
# 2) AHA 17 分区名称与颜色
# =========================
REGION_NAMES = {
    1: "Basal inferoseptal",
    2: "Basal anteroseptal",
    3: "Basal anterior",
    4: "Basal anterolateral",
    5: "Basal inferolateral",
    6: "Basal inferior",
    7: "Mid inferoseptal",
    8: "Mid anteroseptal",
    9: "Mid anterior",
    10: "Mid anterolateral",
    11: "Mid inferolateral",
    12: "Mid inferior",
    13: "Apical septal",
    14: "Apical anterior",
    15: "Apical lateral",
    16: "Apical inferior",
    17: "Apical cap",
}

# 17 个高区分度颜色（适合离散类别）
# 这里使用一组对比度较高、印刷效果较稳的颜色
REGION_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78",
    "#98df8a", "#ff9896", "#c5b0d5", "#c49c94", "#f7b6d2",
]

AHA_CMAP = ListedColormap(REGION_COLORS, name="aha17")
AHA_ANNOTATIONS = {float(i): str(i) for i in range(1, 18)}


In [ ]:

# =========================
# 3) 读取 .mat 和 .inp 的辅助函数
# =========================
def _load_mat_fallback_v73(mat_file):
    '''读取 MATLAB v7.3 (HDF5) 格式的 .mat 文件。'''
    if h5py is None:
        raise RuntimeError(
            "无法用 scipy.io.loadmat 读取该 .mat 文件，且当前环境没有 h5py。"
        )

    with h5py.File(mat_file, "r") as f:
        if "AHALVMeshDivision" not in f:
            raise KeyError("MAT 文件中没有找到 'AHALVMeshDivision' 结构体。")

        grp = f["AHALVMeshDivision"]

        def _read_array(name):
            arr = np.array(grp[name]).squeeze()
            return arr.reshape(-1)

        node_regions = _read_array("nodeRegions").astype(np.int32)
        el_regions = _read_array("elRegions").astype(np.int32)
        return node_regions, el_regions


def load_regions_from_mat(mat_file):
    '''读取 nodeRegions 和 elRegions。'''
    try:
        mat = loadmat(mat_file, squeeze_me=True, struct_as_record=False)
        if "AHALVMeshDivision" not in mat:
            raise KeyError("MAT 文件中没有找到 'AHALVMeshDivision'。")

        s = mat["AHALVMeshDivision"]

        if hasattr(s, "nodeRegions"):
            node_regions = np.asarray(s.nodeRegions).reshape(-1).astype(np.int32)
            el_regions = np.asarray(s.elRegions).reshape(-1).astype(np.int32)
            return node_regions, el_regions

        # 兼容保守访问方式
        node_regions = np.asarray(s["nodeRegions"]).reshape(-1).astype(np.int32)
        el_regions = np.asarray(s["elRegions"]).reshape(-1).astype(np.int32)
        return node_regions, el_regions

    except NotImplementedError:
        return _load_mat_fallback_v73(mat_file)


def parse_abaqus_inp(inp_file):
    '''
    解析 Abaqus .inp 中的：
    - *Node
    - *Element, type=C3D8 or C3D8H

    返回：
    -------
    node_ids : (N,) int
    points   : (N, 3) float
    elem_ids : (M,) int
    conn     : (M, 8) int, 0-based node index
    '''
    node_ids = []
    points = []

    elem_ids = []
    elem_conn_node_ids = []

    in_nodes = False
    in_elements = False
    element_is_hex8 = False

    with open(inp_file, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("**"):
                continue

            if line.startswith("*"):
                upper = line.upper()

                in_nodes = upper.startswith("*NODE")
                in_elements = upper.startswith("*ELEMENT")
                element_is_hex8 = False

                if in_elements and ("TYPE=C3D8" in upper or "TYPE=C3D8H" in upper):
                    element_is_hex8 = True
                continue

            if in_nodes:
                parts = [p.strip() for p in line.split(",") if p.strip()]
                if len(parts) >= 4:
                    node_ids.append(int(parts[0]))
                    points.append([float(parts[1]), float(parts[2]), float(parts[3])])

            elif in_elements and element_is_hex8:
                parts = [p.strip() for p in line.split(",") if p.strip()]
                if len(parts) < 9:
                    raise ValueError(f"六面体单元连通性读取失败，当前行：{line}")
                elem_ids.append(int(parts[0]))
                elem_conn_node_ids.append([int(x) for x in parts[1:9]])

    node_ids = np.asarray(node_ids, dtype=np.int64)
    points = np.asarray(points, dtype=np.float64)
    elem_ids = np.asarray(elem_ids, dtype=np.int64)
    elem_conn_node_ids = np.asarray(elem_conn_node_ids, dtype=np.int64)

    if node_ids.size == 0:
        raise RuntimeError("没有从 .inp 中解析到节点。")
    if elem_ids.size == 0:
        raise RuntimeError("没有从 .inp 中解析到 C3D8/C3D8H 单元。")

    # Abaqus 节点编号 -> 0-based point index
    id_to_idx = {nid: i for i, nid in enumerate(node_ids.tolist())}
    conn = np.empty_like(elem_conn_node_ids, dtype=np.int64)

    for i in range(elem_conn_node_ids.shape[0]):
        for j in range(8):
            conn[i, j] = id_to_idx[elem_conn_node_ids[i, j]]

    return node_ids, points, elem_ids, conn


In [ ]:

# =========================
# 4) 构造 PyVista 网格
# =========================
def build_unstructured_grid(points, conn, node_regions, el_regions):
    '''根据点坐标、六面体连通性和标签，构造 PyVista UnstructuredGrid。'''
    if len(points) != len(node_regions):
        raise ValueError(
            f"节点数不一致：mesh 有 {len(points)} 个节点，但 nodeRegions 长度为 {len(node_regions)}。"
        )
    if len(conn) != len(el_regions):
        raise ValueError(
            f"单元数不一致：mesh 有 {len(conn)} 个单元，但 elRegions 长度为 {len(el_regions)}。"
        )

    n_cells = conn.shape[0]

    # PyVista / VTK 的 cells 扁平格式：
    # [8, p0, p1, ..., p7, 8, p0, ..., p7, ...]
    cells = np.hstack(
        [np.full((n_cells, 1), 8, dtype=np.int64), conn.astype(np.int64)]
    ).ravel()

    celltypes = np.full(n_cells, pv.CellType.HEXAHEDRON, dtype=np.uint8)

    grid = pv.UnstructuredGrid(cells, celltypes, points)
    grid.cell_data["AHA17"] = el_regions.astype(np.int32)
    grid.point_data["AHA17_nodes"] = node_regions.astype(np.int32)

    return grid


def maybe_save_vtu(grid, out_path):
    '''可选：保存为 .vtu，便于后续 ParaView / VTK 工作流复用。'''
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    grid.save(out_path)
    return out_path


In [ ]:

# =========================
# 5) 学术风格渲染辅助函数
# =========================
def compute_camera(bounds):
    '''根据模型 bounds 生成较稳的固定相机。'''
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    center = np.array([(xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2], dtype=float)

    extents = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
    diag = np.linalg.norm(extents)

    # 斜前上方观察，比较适合 LV 这种细长结构
    position = center + np.array([1.35, -1.85, 1.05]) * diag
    viewup = (0.0, 0.0, 1.0)
    return [tuple(position), tuple(center), viewup]


def add_mesh_with_aha_style(
    plotter,
    mesh,
    title,
    show_scalar_bar=False,
    show_edges=False,
):
    '''统一的显示样式。'''
    scalar_bar_args = dict(
        title="AHA 17 segment",
        n_labels=17,
        italic=False,
        bold=False,
        vertical=True,
        position_x=0.88,
        position_y=0.12,
        width=0.08,
        height=0.76,
        fmt="%.0f",
        title_font_size=16,
        label_font_size=12,
        color="black",
    )

    actor = plotter.add_mesh(
        mesh,
        scalars="AHA17",
        preference="cell",
        cmap=AHA_CMAP,
        clim=(0.5, 17.5),
        categories=True,
        annotations=AHA_ANNOTATIONS,
        show_scalar_bar=show_scalar_bar,
        scalar_bar_args=scalar_bar_args,
        interpolate_before_map=False,
        show_edges=show_edges,
        edge_color="#333333",
        line_width=0.7,
        smooth_shading=False,
        ambient=0.15,
        diffuse=0.85,
        specular=0.05,
    )

    plotter.add_text(
        title,
        position="upper_left",
        font_size=16,
        color="black",
        shadow=False,
    )
    return actor


def render_dual_panel_figure(grid, screenshot_path, window_size=(2400, 1100), image_scale=2):
    '''渲染双视图图像并保存。'''
    # 外表面
    surface = grid.extract_surface()

    # 纵向裁切，保留一半体积，再取表面
    center = np.array(grid.center)
    clipped = grid.clip(normal=(1.0, 0.0, 0.0), origin=center, invert=False)
    clipped_surface = clipped.extract_surface()

    # 创建离屏 plotter：更稳，适合 notebook
    pl = pv.Plotter(shape=(1, 2), off_screen=True, window_size=window_size, border=False)
    pl.set_background("white")

    if parallel_projection:
        pl.enable_parallel_projection()

    camera = compute_camera(grid.bounds)

    # 左：整体外表面
    pl.subplot(0, 0)
    add_mesh_with_aha_style(
        pl,
        surface,
        title="Exterior surface",
        show_scalar_bar=True,
        show_edges=show_edges_left,
    )
    pl.camera_position = camera
    pl.camera.zoom(1.1)

    # 右：裁切视图
    pl.subplot(0, 1)
    add_mesh_with_aha_style(
        pl,
        clipped_surface,
        title="Cutaway view",
        show_scalar_bar=False,
        show_edges=show_edges_right,
    )
    pl.camera_position = camera
    pl.camera.zoom(1.1)

    screenshot_path = Path(screenshot_path)
    screenshot_path.parent.mkdir(parents=True, exist_ok=True)

    img = pl.screenshot(filename=str(screenshot_path), scale=image_scale)
    pl.close()

    return img


In [ ]:

# =========================
# 6) 读取数据并构造网格
# =========================
inp_path = Path(inp_path)
mat_path = Path(mat_path)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

figure_path = output_dir / figure_name
vtu_path = output_dir / vtu_name

node_regions, el_regions = load_regions_from_mat(mat_path)
node_ids, points, elem_ids, conn = parse_abaqus_inp(inp_path)

grid = build_unstructured_grid(points, conn, node_regions, el_regions)

print(f"Mesh points : {grid.n_points}")
print(f"Mesh cells  : {grid.n_cells}")
print(f"AHA labels  : {np.unique(grid.cell_data['AHA17'])}")

if save_vtu:
    maybe_save_vtu(grid, vtu_path)
    print(f"Saved VTU   : {vtu_path}")


In [ ]:

# =========================
# 7) 打印 AHA 编号与名称对照
# =========================
for k in range(1, 18):
    print(f"{k:>2d}  ->  {REGION_NAMES[k]}")


In [ ]:

# =========================
# 8) 生成高分辨率静态图
# =========================
img = render_dual_panel_figure(
    grid,
    screenshot_path=figure_path,
    window_size=window_size,
    image_scale=image_scale,
)

print(f"Saved figure: {figure_path}")

plt.figure(figsize=(16, 7))
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()



## 可选：单独查看裁切后的几何

如果你想进一步观察内部结构，可以额外运行下面这格。


In [ ]:

# 可选：单独查看 cutaway，并用更大的窗口导出
cutaway_path = output_dir / "lv_aha17_cutaway_only.png"

clipped = grid.clip(normal=(1.0, 0.0, 0.0), origin=grid.center, invert=False)
clipped_surface = clipped.extract_surface()

pl = pv.Plotter(off_screen=True, window_size=(1400, 1200))
pl.set_background("white")
if parallel_projection:
    pl.enable_parallel_projection()

add_mesh_with_aha_style(
    pl,
    clipped_surface,
    title="Cutaway only",
    show_scalar_bar=True,
    show_edges=True,
)

pl.camera_position = compute_camera(grid.bounds)
pl.camera.zoom(1.15)

img2 = pl.screenshot(filename=str(cutaway_path), scale=2)
pl.close()

print(f"Saved figure: {cutaway_path}")

plt.figure(figsize=(9, 8))
plt.imshow(img2)
plt.axis("off")
plt.tight_layout()
plt.show()



## 使用说明

你通常只需要改第 1 个配置代码块中的这些内容：

- `inp_path`
- `mat_path`
- `output_dir`

运行顺序建议：

1. 依次运行前 8 个单元格
2. 检查输出的 `VTU` 和 PNG
3. 如需单独看裁切图，再运行最后一个可选单元格

## 备注

- 该代码默认按 `elRegions` 进行 **cell-wise** 着色，这是最适合材料区域或分区显示的方式。
- 如果未来你要做更细的 35 分区，可把 `elRegions` 换成 `elRegionsFull` 再重复同样流程。
